<div dir="rtl">

# جلسه ۲۶: پاک‌سازی و آماده‌سازی داده
## Data Cleaning & Preparation

**بخش ۳: محاسبات علمی و کار با داده** — دوره‌ی پایتون برای علوم داده

مدرس: دکتر حسین حق‌بین

</div>

<div dir="rtl">

## 🎯 اهداف جلسه

در پایان این جلسه باید بتوانید:

- مقدارهای **گمشده** را پیدا و بشمارید و بین حذف، پر کردن با یک مقدار، پر کردن **گروهی** و ستون نشانگر، آگاهانه انتخاب کنید،
- داده‌های **پرت** را با قاعده‌ی IQR و z-score پیدا کنید و بدانید چرا z-score گاهی پرت‌ها را «نمی‌بیند»،
- ستون‌های متنیِ کثیف را با <code dir="ltr">pd.to&#95;numeric</code>، <code dir="ltr">pd.to&#95;datetime</code>، <code dir="ltr">astype</code> و متدهای <code dir="ltr">str</code> به نوع درست تبدیل کنید،
- متغیرهای **رسته‌ای** را با one-hot و نگاشت ترتیبی به عدد تبدیل کنید و دام‌های آن را بشناسید،
- داده را **نرمال‌سازی** و **استانداردسازی** کنید و بدانید چرا آماره‌ها فقط باید از داده‌ی آموزش حساب شوند (نشت داده).

> 📖 روش خواندن این جزوه و معنی علامت‌ها (💡 🔍 ❓ ⚠️ 📌 📝) در ابتدای جلسه‌ی ۱ توضیح داده شده است.

</div>

<div dir="rtl">

## فهرست مطالب

1. داده‌های گمشده
2. داده‌های پرت
3. تبدیل نوع
4. کدگذاری متغیرهای رسته‌ای
5. نرمال‌سازی و استانداردسازی

</div>

<div dir="rtl">

### داده‌ی این جلسه

در این جلسه از داده‌ی واقعی **پنگوئن‌های پالمر** استفاده می‌کنیم: اندازه‌های ۳۴۴ پنگوئن از سه گونه در قطب جنوب (منتشرشده با مجوز آزاد CC0). فایل آن در پوشه‌ی <code dir="ltr">Notebooks/data</code> مخزن است. کد زیر اگر فایل را کنار نوت‌بوک پیدا کند از همان استفاده می‌کند، وگرنه آن را از سایت درس می‌خواند:

</div>

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

LOCAL = Path("data/penguins.csv")
URL = "https://haghbinh.github.io/Python-Course/Notebooks/data/penguins.csv"
penguins = pd.read_csv(LOCAL if LOCAL.exists() else URL)

print(penguins.shape)
print(penguins.head())

<div dir="rtl">

🔍 هر سطر یک پنگوئن است: گونه (<code dir="ltr">species</code>)، جزیره (<code dir="ltr">island</code>)، طول و عمق نوک (<code dir="ltr">bill&#95;length&#95;mm</code>، <code dir="ltr">bill&#95;depth&#95;mm</code>)، طول باله (<code dir="ltr">flipper&#95;length&#95;mm</code>)، وزن (<code dir="ltr">body&#95;mass&#95;g</code>)، جنس (<code dir="ltr">sex</code>) و سال اندازه‌گیری. در سطر چهارم (اندیس <code dir="ltr">3</code>) همه‌ی اندازه‌ها <code dir="ltr">NaN</code> هستند؛ یعنی داده‌ی واقعی هم بی‌نقص نیست.

</div>

<div dir="rtl">

## ۱. داده‌های گمشده

### 💡 مفهوم

در Pandas، مقدار گمشده با <code dir="ltr">NaN</code> (برای عددها) یا <code dir="ltr">None</code> نمایش داده می‌شود و هر دو را <code dir="ltr">isna</code> تشخیص می‌دهد. قدم اول همیشه **اندازه‌گیری** است: کدام ستون‌ها، چند مقدار، و آیا گمشدگی الگو دارد؟

| کار | کد |
|---|---|
| ماتریس درست و نادرست | <code dir="ltr">df.isna()</code> / <code dir="ltr">df.notna()</code> |
| تعداد گمشده در هر ستون | <code dir="ltr">df.isna().sum()</code> |
| درصد گمشده در هر ستون | <code dir="ltr">df.isna().mean()</code> |
| سطرهایی که دست‌کم یک گمشده دارند | <code dir="ltr">df[df.isna().any(axis=1)]</code> |

</div>

In [ ]:
print(penguins.isna().sum())
print((penguins.isna().mean() * 100).round(1))
print("rows with any missing:", penguins.isna().any(axis=1).sum())

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">isna()</code> یک DataFrame از <code dir="ltr">True</code>/<code dir="ltr">False</code> می‌سازد و <code dir="ltr">sum()</code> روی هر ستون، <code dir="ltr">True</code>ها را می‌شمارد (چون <code dir="ltr">True</code> مثل <code dir="ltr">1</code> جمع می‌شود؛ جلسه‌ی ۲).
- <code dir="ltr">mean()</code> روی همان ماتریس، **کسر** گمشده‌ها را می‌دهد: ستون <code dir="ltr">sex</code> حدود ۳٫۲٪ گمشده دارد و چهار ستون اندازه هر کدام دو مقدار.
- <code dir="ltr">any(axis=1)</code> برای هر **سطر** می‌پرسد: آیا دست‌کم یک <code dir="ltr">True</code> دارد؟ ۱۱ سطر چنین‌اند.

### 💡 راه اول: حذف با <code dir="ltr">dropna</code>

| آرگومان | معنی |
|---|---|
| (بدون آرگومان) | هر سطری که **هر** مقدار گمشده‌ای دارد حذف می‌شود |
| <code dir="ltr">how="all"</code> | فقط سطرهایی که **همه‌ی** مقدارهایشان گمشده است |
| <code dir="ltr">subset=[...]</code> | فقط گمشدگی در ستون‌های مشخص مهم است |
| <code dir="ltr">thresh=k</code> | سطری می‌ماند که دست‌کم <code dir="ltr">k</code> مقدار **غیرگمشده** داشته باشد |

</div>

In [ ]:
print(penguins.dropna().shape)
print(penguins.dropna(how="all").shape)
print(penguins.dropna(subset=["body_mass_g"]).shape)
print(penguins.dropna(thresh=7).shape)

<div dir="rtl">

🔍 حذف همه‌ی سطرهای ناقص ۱۱ پنگوئن را دور می‌اندازد، در حالی که ۹ تا از آن‌ها فقط <code dir="ltr">sex</code> را ندارند و اندازه‌هایشان کامل است. اگر سؤال ما درباره‌ی وزن باشد، <code dir="ltr">subset=["body&#95;mass&#95;g"]</code> فقط دو سطر واقعاً بی‌مصرف را حذف می‌کند. <code dir="ltr">how="all"</code> چیزی حذف نمی‌کند، چون ستون‌های <code dir="ltr">species</code> و <code dir="ltr">island</code> هیچ‌وقت خالی نیستند. <code dir="ltr">thresh=7</code> یعنی از ۸ ستون دست‌کم ۷ مقدار موجود باشد؛ همان دو سطری که پنج مقدار ندارند حذف می‌شوند.

### ⚠️ اشتباه رایج: <code dir="ltr">dropna</code> بی‌فکر

اگر گمشدگی **تصادفی** نباشد (مثلاً جنس جوجه‌ها همیشه ثبت نشده باشد)، حذف سطرها داده را **سوگیر** می‌کند: نمونه‌ی باقی‌مانده دیگر نماینده‌ی جامعه نیست. قبل از حذف، بپرسید **چرا** این مقدارها گمشده‌اند.

### 💡 راه دوم: پر کردن با <code dir="ltr">fillna</code>

- مقدار ثابت: برای ستون رسته‌ای، یک برچسب مثل <code dir="ltr">"unknown"</code> معمولاً صادقانه‌تر از حدس است.
- میانگین یا **میانه**: برای ستون عددی؛ میانه در برابر داده‌های پرت مقاوم‌تر است.
- **مُد** (پرتکرارترین مقدار): برای ستون رسته‌ای، اگر واقعاً بخواهید حدس بزنید.

</div>

In [ ]:
df = penguins.copy()
df["sex"] = df["sex"].fillna("unknown")
mass_median = df["body_mass_g"].median()
df["body_mass_g"] = df["body_mass_g"].fillna(mass_median)

print(mass_median)
print(df["sex"].value_counts())
print(df["body_mass_g"].isna().sum())

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">penguins.copy()</code> لازم است تا داده‌ی اصلی دست‌نخورده بماند (همان مسئله‌ی نام مستعار در جلسه‌ی ۸).
- <code dir="ltr">fillna</code> یک Series **جدید** برمی‌گرداند؛ باید آن را در ستون انتساب کنید (مثل متدهای رشته در جلسه‌ی ۱۱).
- میانه‌ی وزن <code dir="ltr">4050</code> گرم است؛ دو پنگوئنِ بی‌اندازه همین وزن را می‌گیرند، بی‌توجه به اینکه از کدام گونه‌اند.

### 💡 پر کردن گروهی با <code dir="ltr">transform</code>

وزن پنگوئن‌های Gentoo حدود ۱۳۰۰ گرم بیشتر از دو گونه‌ی دیگر است. پر کردن با میانه‌ی **کل** داده، برای یک Gentoo عددی بسیار کم می‌گذارد. بهتر است هر مقدار گمشده با میانه‌ی **گروه خودش** پر شود. <code dir="ltr">groupby(...).transform("median")</code> (جلسه‌ی ۲۴) یک Series هم‌اندازه با داده برمی‌گرداند که در هر سطر، میانه‌ی گروهِ همان سطر است:

</div>

In [ ]:
group_median = penguins.groupby("species")["body_mass_g"].transform("median")
print(group_median.loc[[3, 271]])

df2 = penguins.copy()
df2["body_mass_missing"] = df2["body_mass_g"].isna()
df2["body_mass_g"] = df2["body_mass_g"].fillna(group_median)
print(df2.loc[[3, 271], ["species", "body_mass_g", "body_mass_missing"]])

<div dir="rtl">

🔍 پنگوئن شماره‌ی <code dir="ltr">3</code> از گونه‌ی Adelie است و میانه‌ی گروه آن (<code dir="ltr">3700</code>) را می‌گیرد؛ پنگوئن <code dir="ltr">271</code> از گونه‌ی Gentoo است و <code dir="ltr">5000</code> می‌گیرد. ستون <code dir="ltr">body&#95;mass&#95;missing</code> یک **ستون نشانگر** است: اطلاعاتِ «این مقدار حدسی است» از بین نمی‌رود و یک مدل می‌تواند از خودِ گمشدگی هم استفاده کند.

### 💡 سری زمانی: پر کردن با مقدار قبلی

وقتی داده‌ها ترتیب زمانی دارند (مثلاً قرائت یک حسگر در هر ساعت)، منطقی‌ترین حدس برای مقدار گمشده اغلب **آخرین مقدار دیده‌شده** است: <code dir="ltr">ffill</code> (forward fill). (سری‌های زمانی را در جلسه‌ی ۲۵ دیدید.)

### ❓ پیش‌بینی کن

</div>

In [ ]:
readings = pd.Series([20.0, np.nan, np.nan, 23.0, np.nan, 25.0])
print(readings.ffill().tolist())
print(readings.fillna(readings.mean()).tolist())
print(readings.interpolate().tolist())

<div dir="rtl">

🔍 <code dir="ltr">ffill</code> هر جای خالی را با آخرین مقدار قبل از آن پر می‌کند: <code dir="ltr">[20, 20, 20, 23, 23, 25]</code>. میانگین سه مقدار موجود <code dir="ltr">22.67</code> است و همه‌ی جاهای خالی همین را می‌گیرند؛ روند زمانی از بین می‌رود. <code dir="ltr">interpolate</code> روی خط راست بین دو همسایه درون‌یابی می‌کند: <code dir="ltr">[20, 21, 22, 23, 24, 25]</code>.

### ⚠️ اشتباه رایج: <code dir="ltr">bfill</code> یا درون‌یابی در پیش‌بینی

<code dir="ltr">bfill</code> (پر کردن با مقدار **بعدی**) و <code dir="ltr">interpolate</code> از آینده استفاده می‌کنند. اگر مدل قرار است در لحظه‌ی <code dir="ltr">t</code> پیش‌بینی کند، در آن لحظه مقدار بعدی را ندارد؛ این کار نوعی **نشت داده** است.

</div>

<div dir="rtl">

## ۲. داده‌های پرت

### 💡 مفهوم

داده‌ی **پرت (outlier)** مقداری است که با بقیه‌ی داده‌ها فاصله‌ی زیادی دارد. ممکن است **خطا** باشد (حسگر خراب، جا افتادن ممیز) یا یک مقدار **واقعی اما نادر** (یک خانه‌ی بسیار گران). دو قاعده‌ی رایج برای پیدا کردن آن:

- **قاعده‌ی IQR:** چارک اول <code dir="ltr">Q1</code> و سوم <code dir="ltr">Q3</code> را حساب کنید و <code dir="ltr">IQR = Q3 - Q1</code>. هر مقدار کمتر از <code dir="ltr">Q1 - 1.5 × IQR</code> یا بیشتر از <code dir="ltr">Q3 + 1.5 × IQR</code> پرت است. نمودار جعبه‌ای (boxplot) دقیقاً همین را نشان می‌دهد (جلسه‌های ۲۷ و ۲۸).
- **قاعده‌ی z-score:** <code dir="ltr">z = (x - mean) / std</code>؛ مقدارهایی با <code dir="ltr">&#124;z&#124; &gt; 3</code> (گاهی ۲ یا ۲٫۵) پرت‌اند.

</div>

<p align="center"><img src="images/s26_iqr_rule.png" width="760" alt="s26_iqr_rule.png"></p>

<p align="center" dir="rtl"><i>قاعده‌ی IQR: جعبه از <code dir="ltr">Q1</code> تا <code dir="ltr">Q3</code> است و هر نقطه بیرون از مرزهای <code dir="ltr">1.5 × IQR</code> پرت حساب می‌شود.</i></p>

In [ ]:
temps = pd.Series([21.5, 22.0, 21.8, 22.3, 21.9, 85.0, 22.1, 21.7, 22.4, -40.0, 22.0, 21.6])

q1, q3 = temps.quantile(0.25), temps.quantile(0.75)
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(round(low, 2), round(high, 2))
print(temps[(temps < low) | (temps > high)].tolist())

z = (temps - temps.mean()) / temps.std()
print(z.round(2).tolist())
print(temps[z.abs() > 3].tolist())

<div dir="rtl">

### 🔍 کد را بخوان

- قرائت‌های یک دماسنج اتاق حدود ۲۲ درجه‌اند، اما دو مقدار <code dir="ltr">85</code> و <code dir="ltr">-40</code> آشکارا خطای حسگرند.
- قاعده‌ی IQR مرزهای <code dir="ltr">20.96</code> و <code dir="ltr">22.86</code> را می‌دهد و هر دو مقدار خراب را پیدا می‌کند.
- اما قاعده‌ی z-score با آستانه‌ی ۳ **هیچ** پرتی پیدا نمی‌کند! z دو مقدار خراب حدود <code dir="ltr">2.36</code> و <code dir="ltr">-2.33</code> است.

### ⚠️ اثر پوشاندن (masking)

z-score از میانگین و انحراف معیار استفاده می‌کند و همین دو آماره از خودِ پرت‌ها تأثیر می‌گیرند. دو مقدار خراب، انحراف معیار را از حدود ۰٫۳ به حدود ۲۶٫۷ رسانده‌اند؛ با این مقیاس بزرگ، خودشان «عادی» به نظر می‌رسند. چارک‌ها **مقاوم (robust)** هستند: چند مقدار افراطی آن‌ها را تکان نمی‌دهد. برای داده‌های کوچک یا پرت‌های زیاد، قاعده‌ی IQR (یا z-score با میانه) قابل اعتمادتر است.

### 💡 با پرت‌ها چه کنیم؟

| روش | کد | کی؟ |
|---|---|---|
| حذف سطر | <code dir="ltr">df[mask]</code> | وقتی مطمئنید خطاست و تعدادش کم است |
| تبدیل به گمشده | <code dir="ltr">s.where(mask)</code> | وقتی خطاست ولی بقیه‌ی سطر ارزشمند است؛ بعد مثل بخش ۱ |
| بریدن (clip / winsorize) | <code dir="ltr">s.clip(low, high)</code> | وقتی مقدار واقعی است ولی نباید روی مدل حاکم شود |
| نگه داشتن | — | وقتی پرت همان چیزی است که دنبالش هستید (کلاه‌برداری، خرابی دستگاه) |

</div>

In [ ]:
cleaned = temps.where((temps >= low) & (temps <= high))
print(cleaned.tolist())
print(temps.clip(low, high).round(2).tolist())

<div dir="rtl">

🔍 <code dir="ltr">where</code> مقدارهایی را که شرط برایشان برقرار **نیست** به <code dir="ltr">NaN</code> تبدیل می‌کند؛ حالا می‌توان آن‌ها را مثلاً با <code dir="ltr">ffill</code> پر کرد. <code dir="ltr">clip</code> به جای حذف، هر مقدار را به مرز نزدیک‌ترش می‌برد: <code dir="ltr">85</code> به <code dir="ltr">22.86</code> و <code dir="ltr">-40</code> به <code dir="ltr">20.96</code>.

### ❓ پیش‌بینی کن

کدام پنگوئن‌ها در وزن، طبق قاعده‌ی IQR روی **کل** داده، پرت‌اند؟ اگر قاعده را **جداگانه برای هر گونه** اجرا کنید، آیا نتیجه فرق می‌کند؟

</div>

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

mass = penguins["body_mass_g"]
print("overall:", iqr_outliers(mass).sum())
per_species = penguins.groupby("species")["body_mass_g"].transform(iqr_outliers)
print("per species:", per_species.sum())
print(penguins.loc[per_species, ["species", "sex", "body_mass_g"]])

<div dir="rtl">

🔍 روی کل داده هیچ پنگوئنی پرت نیست، چون پراکندگی بین گونه‌ها (Gentoo سنگین و دو گونه‌ی دیگر سبک‌تر) بازه را بزرگ کرده است. در داخل هر گونه اما دو پنگوئن Chinstrap پرت‌اند (یکی بسیار سبک و یکی بسیار سنگین). «پرت بودن» همیشه **نسبت به یک گروه مرجع** است. این دو پنگوئن به احتمال زیاد واقعی‌اند؛ پس دلیلی برای حذفشان نیست.

</div>

<div dir="rtl">

## ۳. تبدیل نوع

### 💡 مفهوم

داده‌ای که از فرم‌ها، فایل‌های Excel یا وب می‌آید، اغلب عددها و تاریخ‌ها را به شکل **متن** دارد: با کاما برای هزارگان، فاصله‌ی اضافه، <code dir="ltr">"N/A"</code> یا حتی متن بی‌ربط. تا وقتی نوع ستون متنی (<code dir="ltr">object</code> یا <code dir="ltr">str</code>) است، نه می‌توانید میانگین بگیرید و نه مقایسه کنید. ابزارها:

| کار | کد |
|---|---|
| متن به عدد، مقدار نامعتبر به <code dir="ltr">NaN</code> | <code dir="ltr">pd.to&#95;numeric(s, errors="coerce")</code> |
| متن به تاریخ | <code dir="ltr">pd.to&#95;datetime(s, errors="coerce")</code> |
| تغییر نوع وقتی مطمئنید | <code dir="ltr">s.astype(int)</code>، <code dir="ltr">s.astype("category")</code> |
| پاک‌سازی متن | <code dir="ltr">s.str.strip()</code>، <code dir="ltr">s.str.lower()</code>، <code dir="ltr">s.str.replace(",", "")</code> |

یک داده‌ی کوچکِ عمداً کثیف از آگهی‌های فروش خانه (قیمت به میلیون تومان):

</div>

In [ ]:
listings = pd.DataFrame({
    "city":   [" Bushehr", "bushehr", "Shiraz ", "SHIRAZ", "Tehran", "tehran", "Shiraz", "Bushehr"],
    "area":   ["85", "120", " 1,200 ", "95", "N/A", "abc", "110", "70"],
    "rooms":  ["2", "3", "3", "-1", "2", "2", "", "1"],
    "price":  ["4,250", "6,100", "5,900", "5,100", "9,800", "8,750", "", "3,300"],
    "listed": ["2026-01-05", "2026-01-17", "2026-02-01", "2026-02-10",
               "not known", "2026-03-02", "2026-03-15", "2026-03-20"],
})
print(listings.dtypes)
print(listings["city"].unique())

<div dir="rtl">

🔍 همه‌ی ستون‌ها از نوع **متنی** هستند (در Pandas 2 با نام <code dir="ltr">object</code> و در Pandas 3 با نام <code dir="ltr">str</code> نمایش داده می‌شود) و نام شهرها به شش شکل نوشته شده‌اند، در حالی که فقط سه شهر داریم. یک <code dir="ltr">groupby("city")</code> روی این داده شش گروه می‌سازد.

</div>

In [ ]:
clean = listings.copy()
clean["city"] = clean["city"].str.strip().str.title()
clean["area"] = pd.to_numeric(clean["area"].str.strip().str.replace(",", ""), errors="coerce")
clean["rooms"] = pd.to_numeric(clean["rooms"], errors="coerce")
clean["price"] = pd.to_numeric(clean["price"].str.replace(",", ""), errors="coerce")
clean["listed"] = pd.to_datetime(clean["listed"], errors="coerce")

print(clean.dtypes)
print(clean)
print(clean["city"].value_counts())

<div dir="rtl">

### 🔍 کد را بخوان

- متدهای <code dir="ltr">str</code> همان متدهای رشته‌ی جلسه‌ی ۱۱ هستند که روی **همه‌ی** خانه‌های ستون اعمال می‌شوند؛ زنجیر <code dir="ltr">strip().title()</code> شش شکل را به سه نام یکسان می‌رساند.
- برای <code dir="ltr">area</code> اول فاصله‌ها و کامای هزارگان حذف می‌شود، بعد <code dir="ltr">to&#95;numeric</code>. با <code dir="ltr">errors="coerce"</code>، مقدارهای <code dir="ltr">"N/A"</code> و <code dir="ltr">"abc"</code> به جای خطا، <code dir="ltr">NaN</code> می‌شوند (همان ایده‌ی <code dir="ltr">try</code>/<code dir="ltr">except</code> جلسه‌ی ۱۵، برای یک ستون کامل).
- رشته‌ی خالی در <code dir="ltr">rooms</code> و <code dir="ltr">price</code> هم <code dir="ltr">NaN</code> می‌شود. نوع <code dir="ltr">rooms</code> به <code dir="ltr">float64</code> تبدیل شده، چون نوع <code dir="ltr">int64</code> نمی‌تواند <code dir="ltr">NaN</code> نگه دارد.
- <code dir="ltr">"not known"</code> در تاریخ به <code dir="ltr">NaT</code> (معادل <code dir="ltr">NaN</code> برای زمان) تبدیل شده است.

### ⚠️ اشتباه رایج: تبدیل موفق یعنی داده‌ی درست نیست

تبدیل نوع فقط **قالب** را درست می‌کند، نه **معنا** را. هنوز دو مشکل باقی است: خانه‌ای با <code dir="ltr">-1</code> اتاق (ناممکن) و خانه‌ای با مساحت <code dir="ltr">1200</code> متر که با قیمتش جور نیست (احتمالاً <code dir="ltr">120</code> بوده). این‌ها را با **قاعده‌های دامنه** (مثلاً «تعداد اتاق باید دست‌کم ۱ باشد») و قاعده‌های پرت (بخش ۲) پیدا می‌کنیم:

</div>

In [ ]:
clean.loc[clean["rooms"] < 1, "rooms"] = np.nan
print(clean[["city", "area", "rooms", "price"]])
print(clean.isna().sum())

<div dir="rtl">

🔍 مقدار ناممکن به گمشده تبدیل شد؛ حالا تصمیم‌گیری درباره‌ی آن، همان تصمیم‌های بخش ۱ است. شمارش نهایی گمشده‌ها: دو مساحت، دو تعداد اتاق، یک قیمت و یک تاریخ.

### 💡 نوع <code dir="ltr">category</code>

ستونی که تعداد کمی مقدار متمایز دارد و بارها تکرار می‌شود (گونه، جزیره، جنس)، با نوع <code dir="ltr">category</code> بسیار کم‌حجم‌تر نگه‌داری می‌شود: Pandas هر مقدار متمایز را یک بار ذخیره می‌کند و در هر سطر فقط یک **کد عددی** می‌گذارد.

</div>

In [ ]:
species_obj = penguins["species"]
species_cat = species_obj.astype("category")
print(species_obj.memory_usage(deep=True), species_cat.memory_usage(deep=True))
print(species_cat.cat.categories.tolist())
print(species_cat.cat.codes.head(3).tolist(), species_cat.cat.codes.tail(3).tolist())

<div dir="rtl">

🔍 همین ستون با نوع <code dir="ltr">category</code> حدود ۳۰ برابر کم‌حجم‌تر است (حدود ۱۹ هزار بایت در برابر حدود ۶۰۰ بایت). دسته‌ها به ترتیب الفبا مرتب‌اند (<code dir="ltr">Adelie</code>، <code dir="ltr">Chinstrap</code>، <code dir="ltr">Gentoo</code>) و کدهای <code dir="ltr">0</code>، <code dir="ltr">1</code>، <code dir="ltr">2</code> می‌گیرند؛ سه سطر اول Adelie (کد <code dir="ltr">0</code>) و سه سطر آخر Chinstrap (کد <code dir="ltr">1</code>) هستند. این کدها در بخش ۴ دوباره به کار می‌آیند، با یک هشدار جدی.

</div>

<div dir="rtl">

## ۴. کدگذاری متغیرهای رسته‌ای

### 💡 مفهوم

بیشتر مدل‌های یادگیری ماشین فقط **عدد** می‌فهمند. متغیر رسته‌ای (<code dir="ltr">species</code>، <code dir="ltr">island</code>) را باید به عدد تبدیل کرد، و نحوه‌ی تبدیل مهم است:

- **اسمی (nominal):** دسته‌ها ترتیب ندارند (جزیره‌ها). روش درست: **one-hot**؛ برای هر دسته یک ستون صفر و یکی.
- **ترتیبی (ordinal):** دسته‌ها ترتیب طبیعی دارند (کوچک، متوسط، بزرگ). روش درست: نگاشت به عددهایی که همان ترتیب را حفظ کنند.

</div>

In [ ]:
small = penguins.loc[[0, 152, 300], ["species", "island", "body_mass_g"]]
print(small)
print(pd.get_dummies(small, columns=["species", "island"], dtype=int))

<div dir="rtl">

### 🔍 کد را بخوان

- <code dir="ltr">get&#95;dummies</code> به جای هر ستون رسته‌ای، یک ستون برای **هر دسته** می‌سازد (<code dir="ltr">species&#95;Adelie</code>، <code dir="ltr">species&#95;Chinstrap</code>، ...). در هر سطر دقیقاً یکی از ستون‌های هر گروه <code dir="ltr">1</code> است. ستون‌های عددی (<code dir="ltr">body&#95;mass&#95;g</code>) دست‌نخورده می‌مانند.
- با <code dir="ltr">dtype=int</code> ستون‌ها صفر و یک می‌شوند؛ بدون آن، <code dir="ltr">True</code>/<code dir="ltr">False</code> می‌گیرید.
- این همان کدگذاری one-hot است که برای برچسب‌ها در شبکه‌های عصبی هم استفاده می‌شود (خروجی softmax با یک بردار one-hot مقایسه می‌شود).

### ⚠️ اشتباه رایج: کدهای عددی برای دسته‌های بی‌ترتیب

اگر <code dir="ltr">species</code> را با کدهای <code dir="ltr">0</code>، <code dir="ltr">1</code>، <code dir="ltr">2</code> به مدل خطی بدهید، مدل فرض می‌کند Chinstrap «بین» Adelie و Gentoo است و فاصله‌ی Adelie تا Gentoo دو برابر فاصله‌ی Adelie تا Chinstrap است؛ فرضی کاملاً ساختگی که فقط از ترتیب الفبایی نام‌ها آمده است. برای متغیر اسمی از one-hot استفاده کنید.

### 💡 کدگذاری ترتیبی با <code dir="ltr">map</code>

وقتی دسته‌ها ترتیب واقعی دارند، یک دیکشنری (جلسه‌ی ۱۰) ترتیب را **صریحاً** تعیین می‌کند:

</div>

In [ ]:
size = pd.Series(["medium", "small", "large", "medium", "Large"])
order = {"small": 0, "medium": 1, "large": 2}
print(size.map(order).tolist())
print(size.str.lower().map(order).tolist())

<div dir="rtl">

🔍 <code dir="ltr">map</code> هر مقدار را در دیکشنری جست‌وجو می‌کند؛ مقداری که کلیدش در دیکشنری نیست (<code dir="ltr">"Large"</code> با حرف بزرگ) **بی‌صدا** <code dir="ltr">NaN</code> می‌شود، بدون هیچ خطایی. همیشه قبل از <code dir="ltr">map</code> متن را یکنواخت کنید (بخش ۳) و بعد از آن تعداد <code dir="ltr">NaN</code>ها را بررسی کنید.

### ❓ پیش‌بینی کن

اگر <code dir="ltr">drop&#95;first=True</code> بدهید، ستون‌های one-hot زیر چه می‌شوند؟ چرا یک ستون کمتر، اطلاعاتی را از بین نمی‌برد؟

</div>

In [ ]:
islands = pd.Series(["Biscoe", "Dream", "Torgersen", "Dream"], name="island")
print(pd.get_dummies(islands, dtype=int))
print(pd.get_dummies(islands, drop_first=True, dtype=int))

<div dir="rtl">

🔍 با <code dir="ltr">drop&#95;first=True</code> ستون اولِ الفبایی (<code dir="ltr">Biscoe</code>) حذف می‌شود. اطلاعاتی از بین نمی‌رود: سطری که در هر دو ستون باقی‌مانده <code dir="ltr">0</code> دارد، حتماً <code dir="ltr">Biscoe</code> است. اگر همه‌ی ستون‌ها بمانند، مجموعشان در هر سطر همیشه <code dir="ltr">1</code> است؛ یعنی یک ستون دقیقاً از روی بقیه قابل محاسبه است (**هم‌خطی کامل**). در رگرسیون خطی با عرض از مبدأ، این مسئله برآورد ضرایب را نامعین می‌کند و به آن **دام متغیر مجازی (dummy variable trap)** می‌گویند.

</div>

<div dir="rtl">

## ۵. نرمال‌سازی و استانداردسازی

### 💡 مفهوم

ستون‌های عددی مقیاس‌های بسیار متفاوت دارند: وزن پنگوئن حدود ۴۰۰۰ گرم است و عمق نوک حدود ۱۷ میلی‌متر. در روش‌هایی که **فاصله** حساب می‌کنند (kNN، جلسه‌ی ۱۳) یا با **گرادیان کاهشی** آموزش می‌بینند (جلسه‌ی ۱۸)، ستونی که عددهای بزرگ‌تری دارد بی‌دلیل غالب می‌شود. دو تبدیل رایج:

| نام | فرمول | نتیجه |
|---|---|---|
| نرمال‌سازی (min-max) | <code dir="ltr">(x - min) / (max - min)</code> | بازه‌ی <code dir="ltr">[0, 1]</code> |
| استانداردسازی (z-score) | <code dir="ltr">(x - mean) / std</code> | میانگین <code dir="ltr">0</code> و انحراف معیار <code dir="ltr">1</code> |

</div>

In [ ]:
cols = ["bill_depth_mm", "body_mass_g"]
data = penguins[cols].dropna()

minmax = (data - data.min()) / (data.max() - data.min())
zscore = (data - data.mean()) / data.std()

print(data.describe().loc[["mean", "std", "min", "max"]].round(1))
print(minmax.describe().loc[["min", "max"]].round(2))
print(zscore.describe().loc[["mean", "std"]].round(2))

<div dir="rtl">

### 🔍 کد را بخوان

- عملیات روی DataFrame **ستون به ستون** انجام می‌شود: <code dir="ltr">data.min()</code> یک Series با کمینه‌ی هر ستون است و تفریق، با تطبیق نام ستون‌ها (همان broadcasting جلسه‌ی ۲۲) انجام می‌شود.
- بعد از min-max هر دو ستون دقیقاً بین <code dir="ltr">0</code> و <code dir="ltr">1</code> هستند؛ بعد از z-score هر دو میانگین <code dir="ltr">0</code> و انحراف معیار <code dir="ltr">1</code> دارند. حالا یک واحد تغییر در هر ستون، اهمیت یکسانی دارد.

### ⚠️ اشتباه رایج: آماره‌ها از کل داده

در جلسه‌ی ۱۸ کلاس <code dir="ltr">StandardScaler</code> را ساختید: <code dir="ltr">fit</code> میانگین و انحراف معیار را **فقط از داده‌ی آموزش** یاد می‌گرفت و <code dir="ltr">transform</code> همان عددها را روی هر داده‌ای اعمال می‌کرد. همین قاعده برای همه‌ی کارهای این جلسه برقرار است: میانه‌ای که با آن پر می‌کنید، مرزهای IQR، دسته‌های one-hot، کمینه و بیشینه. اگر این آماره‌ها را از کل داده حساب کنید، اطلاعاتی از داده‌ی آزمون به آموزش **نشت** می‌کند و ارزیابی شما خوش‌بینانه می‌شود. (کتابخانه‌ی scikit-learn برای همین کارها کلاس‌های آماده‌ای مثل <code dir="ltr">StandardScaler</code>، <code dir="ltr">SimpleImputer</code> و <code dir="ltr">OneHotEncoder</code> دارد که دقیقاً با الگوی <code dir="ltr">fit</code> / <code dir="ltr">transform</code> کار می‌کنند.)

</div>

In [ ]:
shuffled = data.sample(frac=1, random_state=0)
n_train = int(0.8 * len(shuffled))
train, test = shuffled.iloc[:n_train], shuffled.iloc[n_train:]

mu, sigma = train.mean(), train.std()
train_z = (train - mu) / sigma
test_z = (test - mu) / sigma

print(len(train), len(test))
print(train_z.mean().round(3).tolist(), test_z.mean().round(3).tolist())

<div dir="rtl">

🔍 <code dir="ltr">sample(frac=1, random&#95;state=0)</code> سطرها را تکرارپذیر به هم می‌ریزد (مثل <code dir="ltr">shuffle</code> با <code dir="ltr">seed</code> در جلسه‌ی ۸). میانگین داده‌ی آموزش استانداردشده دقیقاً صفر است، اما میانگین داده‌ی آزمون **کمی** با صفر فرق دارد. این درست است، نه اشکال: داده‌ی آزمون نماینده‌ی داده‌ای است که مدل هنوز ندیده و آماره‌هایش لزوماً با آموزش یکی نیست.

### ❓ پیش‌بینی کن

اگر در ستونی همه‌ی مقدارهای داده‌ی آموزش برابر باشند، دو فرمول بالا چه می‌دهند؟ (راهنمایی: مخرج‌ها را ببینید.)

<details>
<summary><b>پاسخ</b></summary>

مخرج هر دو فرمول (<code dir="ltr">max - min</code> و <code dir="ltr">std</code>) صفر می‌شود. Pandas خطا نمی‌دهد و نتیجه <code dir="ltr">NaN</code> است (یا برای مقدارهای غیربرابر در داده‌ی آزمون، <code dir="ltr">inf</code>). چنین ستونی هیچ اطلاعاتی برای مدل ندارد و بهتر است قبل از مقیاس‌بندی حذف شود.

</details>

</div>

<div dir="rtl">

## 📌 آنچه باید بلد باشید

- شمردن گمشده‌ها با <code dir="ltr">isna().sum()</code> و <code dir="ltr">isna().mean()</code>؛ پیدا کردن سطرهای ناقص با <code dir="ltr">any(axis=1)</code>.
- رفتار <code dir="ltr">dropna</code> با <code dir="ltr">how</code>، <code dir="ltr">subset</code> و <code dir="ltr">thresh</code>؛ خطر سوگیری وقتی گمشدگی تصادفی نیست.
- <code dir="ltr">fillna</code> با ثابت، میانه و مُد؛ پر کردن گروهی با <code dir="ltr">groupby().transform</code>؛ ستون نشانگر؛ <code dir="ltr">ffill</code> در سری زمانی و چرا <code dir="ltr">bfill</code> نشت دارد.
- قاعده‌ی IQR و z-score؛ اثر پوشاندن؛ پرت نسبت به گروه مرجع؛ <code dir="ltr">where</code> و <code dir="ltr">clip</code>.
- <code dir="ltr">pd.to&#95;numeric</code> و <code dir="ltr">pd.to&#95;datetime</code> با <code dir="ltr">errors="coerce"</code>؛ پاک‌سازی با متدهای <code dir="ltr">str</code>؛ قاعده‌های دامنه؛ نوع <code dir="ltr">category</code> و حجم حافظه.
- one-hot با <code dir="ltr">get&#95;dummies</code> و <code dir="ltr">drop&#95;first</code>؛ دام متغیر مجازی؛ کدگذاری ترتیبی با <code dir="ltr">map</code> و <code dir="ltr">NaN</code>های بی‌صدا؛ چرا کدهای عددی برای دسته‌های اسمی گمراه‌کننده است.
- min-max و z-score؛ محاسبه‌ی همه‌ی آماره‌ها فقط از داده‌ی آموزش.

</div>

<div dir="rtl">

## 📝 سؤالات مرور / نمونه سؤال امتحان

> اول خودتان جواب دهید (روی کاغذ، بدون اجرای کد)، بعد پاسخ را باز کنید و در آخر کد را اجرا کنید.

### سؤال ۱ — <code dir="ltr">dropna</code> و <code dir="ltr">fillna</code>

DataFrame زیر را در نظر بگیرید:

</div>

```python
df = pd.DataFrame({
    "a": [1.0, np.nan, 3.0, np.nan],
    "b": [np.nan, np.nan, 6.0, 8.0],
    "c": ["x", None, "z", "w"],
})
```

<div dir="rtl">

شکل (<code dir="ltr">shape</code>) یا مقدار هر عبارت را بنویسید.

| عبارت | حاصل؟ |
|---|---|
| <code dir="ltr">df.isna().sum().tolist()</code> | |
| <code dir="ltr">df.dropna().shape</code> | |
| <code dir="ltr">df.dropna(how="all").shape</code> | |
| <code dir="ltr">df.dropna(thresh=2).shape</code> | |
| <code dir="ltr">df.dropna(subset=["b"]).shape</code> | |
| <code dir="ltr">df.dropna(axis=1).shape</code> | |
| <code dir="ltr">df["a"].fillna(df["a"].mean()).tolist()</code> | |
| <code dir="ltr">df["b"].ffill().tolist()</code> | |

<details>
<summary><b>پاسخ</b></summary>

| عبارت | حاصل |
|---|---|
| <code dir="ltr">df.isna().sum().tolist()</code> | <code dir="ltr">[2, 2, 1]</code> |
| <code dir="ltr">df.dropna().shape</code> | <code dir="ltr">(1, 3)</code>: فقط سطر <code dir="ltr">2</code> کامل است |
| <code dir="ltr">df.dropna(how="all").shape</code> | <code dir="ltr">(3, 3)</code>: سطر <code dir="ltr">1</code> در هر سه ستون گمشده است |
| <code dir="ltr">df.dropna(thresh=2).shape</code> | <code dir="ltr">(3, 3)</code>: سطرهای <code dir="ltr">0</code> و <code dir="ltr">3</code> دو مقدار دارند و می‌مانند؛ سطر <code dir="ltr">1</code> هیچ مقداری ندارد |
| <code dir="ltr">df.dropna(subset=["b"]).shape</code> | <code dir="ltr">(2, 3)</code> |
| <code dir="ltr">df.dropna(axis=1).shape</code> | <code dir="ltr">(4, 0)</code>: هر سه ستون دست‌کم یک گمشده دارند و ستون‌ها حذف می‌شوند |
| <code dir="ltr">df["a"].fillna(df["a"].mean()).tolist()</code> | <code dir="ltr">[1.0, 2.0, 3.0, 2.0]</code> |
| <code dir="ltr">df["b"].ffill().tolist()</code> | <code dir="ltr">[nan, nan, 6.0, 8.0]</code>: قبل از اولین مقدار، چیزی برای پر کردن نیست |

</details>

### سؤال ۲ — پر کردن با میانگین، بی‌هزینه نیست

دو متغیر داریم و دو مقدار از <code dir="ltr">x</code> گمشده است:

</div>

```python
x = pd.Series([2.0, 4.0, np.nan, 8.0, np.nan, 10.0])
y = pd.Series([1.0, 3.0, 5.0, 7.0, 9.0, 11.0])
x_filled = x.fillna(x.mean())
print(x.mean(), x.var(), x_filled.var())
print(round(x.corr(y), 3), round(x_filled.corr(y), 3))
```

<div dir="rtl">

الف) خروجی را پیش‌بینی کنید. (راهنمایی: <code dir="ltr">var</code> و <code dir="ltr">corr</code> در Pandas مقدارهای گمشده را نادیده می‌گیرند.)

ب) چرا واریانس بعد از پر کردن کم می‌شود؟ این برای یک مدل که از پراکندگی ویژگی‌ها استفاده می‌کند چه پیامدی دارد؟

ج) چرا همبستگی ضعیف‌تر شده است؟ اگر به جای میانگین، <code dir="ltr">x</code> را با <code dir="ltr">y</code> (که کامل است) پیش‌بینی می‌کردیم، انتظار دارید چه شود؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">6.0 13.333... 8.0</code> و <code dir="ltr">0.988 0.907</code>.

ب) دو مقدار جدید دقیقاً روی میانگین قرار گرفته‌اند؛ مجموع مربع انحراف‌ها تغییر نمی‌کند (انحراف آن‌ها صفر است) اما تعداد نمونه‌ها از ۴ به ۶ می‌رسد، پس واریانس (با مخرج <code dir="ltr">n - 1</code>) از <code dir="ltr">40/3</code> به <code dir="ltr">40/5</code> کم می‌شود. داده **کم‌پراکنده‌تر و مطمئن‌تر** از واقعیت به نظر می‌رسد: انحراف معیارها، فاصله‌های اطمینان و هر استانداردسازی بعدی کمی غلط می‌شوند.

ج) در سطرهای <code dir="ltr">2</code> و <code dir="ltr">4</code>، <code dir="ltr">y</code> مقدارهای <code dir="ltr">5</code> و <code dir="ltr">9</code> را دارد اما <code dir="ltr">x</code> در هر دو <code dir="ltr">6</code> گرفته است؛ این دو نقطه از خط رابطه بیرون افتاده‌اند و رابطه را ضعیف‌تر نشان می‌دهند. پر کردن با میانگین، رابطه‌ی متغیرها را **نادیده** می‌گیرد. اگر <code dir="ltr">x</code> را از روی <code dir="ltr">y</code> پیش‌بینی کنیم (مثلاً با رگرسیون خطی روی سطرهای کامل؛ **imputation مبتنی بر مدل**)، همبستگی حفظ می‌شود (حتی کمی بیش از حد قوی می‌شود، چون مقدارهای پرشده هیچ نوفه‌ای ندارند).

</details>

### سؤال ۳ — پر کردن گروهی و نشت داده

دانشجویی برای آماده کردن داده‌ی پنگوئن‌ها این کد را نوشته است:

</div>

```python
full = penguins.copy()
full["body_mass_g"] = full["body_mass_g"].fillna(
    full.groupby("species")["body_mass_g"].transform("mean"))
train = full.sample(frac=0.8, random_state=1)
test = full.drop(train.index)
```

<div dir="rtl">

الف) خط دوم دقیقاً چه می‌کند؟ چرا <code dir="ltr">transform</code> و نه <code dir="ltr">agg</code>؟ (اگر <code dir="ltr">agg("mean")</code> بود چه می‌شد؟)

ب) ترتیب کارها چه ایرادی دارد؟ کد درست را بنویسید که میانگین گروه‌ها **فقط** از داده‌ی آموزش حساب شود و روی هر دو بخش اعمال شود.

ج) اگر در داده‌ی آزمون گونه‌ای باشد که در داده‌ی آموزش اصلاً نبوده، کد شما چه می‌کند؟

<details>
<summary><b>پاسخ</b></summary>

الف) برای هر سطر، میانگین وزن گونه‌ی همان سطر را می‌سازد (یک Series هم‌اندازه با <code dir="ltr">full</code> و با همان اندیس) و فقط جاهای خالی را با آن پر می‌کند. <code dir="ltr">agg("mean")</code> یک Series سه‌عضوی با اندیس نام گونه‌ها برمی‌گرداند؛ <code dir="ltr">fillna</code> آن را بر اساس **اندیس** با سطرها تطبیق می‌دهد، هیچ اندیسی جور نمی‌شود و جاهای خالی پر نمی‌شوند (بدون خطا).

ب) میانگین‌ها از **کل** داده (شامل سطرهایی که بعداً آزمون می‌شوند) حساب شده‌اند: نشت داده. درست:

<div dir="ltr">

```python
train = penguins.sample(frac=0.8, random_state=1)
test = penguins.drop(train.index)

means = train.groupby("species")["body_mass_g"].mean()      # learned on train only

def fill_mass(part):
    part = part.copy()
    part["body_mass_g"] = part["body_mass_g"].fillna(part["species"].map(means))
    return part

train, test = fill_mass(train), fill_mass(test)
```

</div>

<code dir="ltr">part["species"].map(means)</code> برای هر سطر میانگین گونه‌اش را **از روی داده‌ی آموزش** می‌آورد؛ همان الگوی <code dir="ltr">fit</code> / <code dir="ltr">transform</code> جلسه‌ی ۱۸.

ج) <code dir="ltr">map</code> برای گونه‌ی ناشناخته <code dir="ltr">NaN</code> برمی‌گرداند و آن جای خالی پر نمی‌شود. باید یک مقدار پشتیبان داشته باشید، مثلاً میانگین کل داده‌ی آموزش: <code dir="ltr">.fillna(part["species"].map(means)).fillna(train["body&#95;mass&#95;g"].mean())</code>.

</details>

### سؤال ۴ — IQR یا z-score؟

داده‌ی زیر زمان پاسخ (میلی‌ثانیه) یک سرویس است:

</div>

```python
d = pd.Series([10, 11, 12, 11, 10, 12, 11, 100, 105])
```

<div dir="rtl">

الف) <code dir="ltr">Q1</code>، <code dir="ltr">Q3</code>، <code dir="ltr">IQR</code> و مرزهای قاعده‌ی IQR را حساب کنید. کدام مقدارها پرت‌اند؟

ب) z-score دو مقدار بزرگ حدود <code dir="ltr">1.7</code> و <code dir="ltr">1.8</code> است. با آستانه‌ی ۳ (و حتی ۲) چند پرت پیدا می‌شود؟ چرا؟

ج) یک نسخه‌ی مقاوم از z-score پیشنهاد کنید (راهنمایی: به جای میانگین و انحراف معیار، از آماره‌هایی استفاده کنید که پرت‌ها تکانشان نمی‌دهند) و آن را با Pandas بنویسید.

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">Q1 = 11</code>، <code dir="ltr">Q3 = 12</code>، <code dir="ltr">IQR = 1</code>؛ مرزها <code dir="ltr">9.5</code> و <code dir="ltr">13.5</code>. مقدارهای <code dir="ltr">100</code> و <code dir="ltr">105</code> پرت‌اند.

ب) هیچ. دو پرت از ۹ داده، میانگین را به حدود <code dir="ltr">31</code> و انحراف معیار را به حدود <code dir="ltr">40</code> رسانده‌اند؛ با این مقیاس، فاصله‌ی خود پرت‌ها از میانگین کمتر از دو انحراف معیار است (اثر پوشاندن). هرچه پرت‌ها بیشتر باشند، z-score کورتر می‌شود.

ج) **z-score مقاوم** با میانه و انحراف مطلق میانه (MAD):

<div dir="ltr">

```python
med = d.median()
mad = (d - med).abs().median()
robust_z = 0.6745 * (d - med) / mad
print(d[robust_z.abs() > 3.5].tolist())      # [100, 105]
```

</div>

میانه <code dir="ltr">11</code> و MAD برابر <code dir="ltr">1</code> است؛ دو مقدار بزرگ z مقاوم حدود <code dir="ltr">60</code> و <code dir="ltr">63</code> می‌گیرند. ضریب <code dir="ltr">0.6745</code> مقیاس را با انحراف معیار توزیع نرمال هم‌خوان می‌کند و آستانه‌ی <code dir="ltr">3.5</code> یک انتخاب متداول است.

</details>

### سؤال ۵ — min-max و داده‌ی پرت

الف) ستون <code dir="ltr">v = pd.Series([10.0, 20, 30, 40, 1000])</code> را با min-max نرمال کنید. چهار مقدار اول چه می‌شوند؟ این برای مدلی که از این ستون استفاده می‌کند چه معنایی دارد؟

ب) **مقیاس‌بندی مقاوم (robust scaling)** از فرمول <code dir="ltr">(x - median) / IQR</code> استفاده می‌کند. خروجی را برای همین ستون حساب کنید.

ج) مدلی روی داده‌ی آموزش با min-max آموزش دیده است. در داده‌ی آزمون مقداری بزرگ‌تر از بیشینه‌ی آموزش دیده می‌شود. خروجی تبدیل چه بازه‌ای دارد؟ آیا باید <code dir="ltr">min</code> و <code dir="ltr">max</code> را با داده‌ی آزمون دوباره حساب کرد؟

<details>
<summary><b>پاسخ</b></summary>

الف) <code dir="ltr">[0.0, 0.01, 0.02, 0.03, 1.0]</code>. یک مقدار پرت کل بازه را اشغال کرده و چهار مقدار عادی در فاصله‌ی ۰ تا ۰٫۰۳ فشرده شده‌اند؛ تفاوت <code dir="ltr">10</code> و <code dir="ltr">40</code> که در داده‌ی اصلی قابل توجه بود، برای مدل تقریباً نامرئی می‌شود.

ب) میانه <code dir="ltr">30</code>، <code dir="ltr">Q1 = 20</code>، <code dir="ltr">Q3 = 40</code>، <code dir="ltr">IQR = 20</code>: خروجی <code dir="ltr">[-1.0, -0.5, 0.0, 0.5, 48.5]</code>. مقدارهای عادی در مقیاس معقولی می‌مانند و فقط پرت بزرگ است (و می‌توان آن را با <code dir="ltr">clip</code> محدود کرد).

ج) خروجی بزرگ‌تر از <code dir="ltr">1</code> می‌شود؛ min-max تضمینی برای داده‌ی ندیده نمی‌دهد. نه: آماره‌ها **همیشه** از آموزش می‌آیند؛ بازمحاسبه با آزمون یعنی همان مقدار خام در آموزش و آزمون به دو عدد متفاوت تبدیل شود و این نشت داده هم هست. اگر بازه‌ی <code dir="ltr">[0, 1]</code> واقعاً لازم است، خروجی را <code dir="ltr">clip(0, 1)</code> کنید.

</details>

### سؤال ۶ — one-hot در عمل

داده‌ی آموزش و آزمون یک مدل این‌طور است:

</div>

```python
train = pd.DataFrame({"island": ["Biscoe", "Dream", "Biscoe", "Torgersen"]})
test = pd.DataFrame({"island": ["Dream", "Anvers"]})

X_train = pd.get_dummies(train, dtype=int)
X_test = pd.get_dummies(test, dtype=int)
print(X_train.columns.tolist())
print(X_test.columns.tolist())
```

<div dir="rtl">

الف) خروجی را بنویسید. اگر <code dir="ltr">X&#95;test</code> را مستقیماً به مدلی بدهید که روی <code dir="ltr">X&#95;train</code> آموزش دیده، چه مشکلی پیش می‌آید؟

ب) با <code dir="ltr">reindex(columns=..., fill&#95;value=0)</code> این مشکل را حل کنید. پنگوئن جزیره‌ی <code dir="ltr">Anvers</code> در نتیجه چه بردار ویژگی‌ای می‌گیرد؟ این بردار چه معنایی دارد؟

ج) ستونی مثل «کد پستی» ۵۰۰۰ مقدار متمایز دارد که بیشترشان فقط یکی دو بار تکرار شده‌اند. چرا one-hot ساده برای آن مناسب نیست؟ دو راه پیشنهاد کنید.

<details>
<summary><b>پاسخ</b></summary>

الف)

<div dir="ltr">

```text
['island_Biscoe', 'island_Dream', 'island_Torgersen']
['island_Anvers', 'island_Dream']
```

</div>

ستون‌ها در دو جدول نه هم‌تعدادند و نه هم‌ترتیب؛ مدل یا خطای ابعاد می‌دهد، یا (اگر تعداد برابر بود) ستون‌ها را **بی‌صدا** جابه‌جا تفسیر می‌کند. دسته‌ها باید از داده‌ی آموزش «یاد گرفته» شوند، مثل بقیه‌ی آماره‌ها.

ب)

<div dir="ltr">

```python
X_test = pd.get_dummies(test, dtype=int).reindex(columns=X_train.columns, fill_value=0)
```

</div>

نتیجه <code dir="ltr">[[0, 1, 0], [0, 0, 0]]</code>. ستون <code dir="ltr">island&#95;Anvers</code> حذف می‌شود و ستون‌های غایب با <code dir="ltr">0</code> ساخته می‌شوند. پنگوئن <code dir="ltr">Anvers</code> بردار تمام‌صفر می‌گیرد؛ یعنی «هیچ‌کدام از جزیره‌های شناخته‌شده». (اگر در آموزش <code dir="ltr">drop&#95;first=True</code> استفاده شده بود، بردار تمام‌صفر با دسته‌ی حذف‌شده یکی می‌شد و این دو قابل تشخیص نبودند.)

ج) ۵۰۰۰ ستون پراکنده می‌سازد که بیشترشان فقط یکی دو <code dir="ltr">1</code> دارند؛ مدل از آن‌ها چیزی یاد نمی‌گیرد جز حفظ کردن (بیش‌برازش) و حافظه و زمان هدر می‌رود. راه‌ها: (۱) دسته‌های **کم‌تکرار** را در یک دسته‌ی <code dir="ltr">"other"</code> ادغام کنید (مثلاً با <code dir="ltr">value&#95;counts</code> و یک آستانه)؛ (۲) **کدگذاری فراوانی**: هر دسته را با تعداد یا نسبت تکرارش (محاسبه‌شده روی آموزش) جایگزین کنید؛ (۳) از ساختار خود داده استفاده کنید (مثلاً دو رقم اول کد پستی یعنی منطقه)؛ (۴) در شبکه‌های عصبی، **embedding** برای هر دسته یاد گرفته می‌شود.

</details>

### سؤال ۷ — تکمیل کد: یک تابع آماده‌سازی کامل

تابع زیر داده‌ی آموزش و آزمون را می‌گیرد و هر دو را آماده‌ی مدل می‌کند: گمشده‌های عددی با میانه پر می‌شوند، ستون رسته‌ای one-hot می‌شود و ستون‌های عددی استاندارد می‌شوند. **همه‌ی** آماره‌ها باید از <code dir="ltr">train</code> بیایند. جاهای خالی را کامل کنید.

</div>

```python
def prepare(train, test, num_cols, cat_col):
    train, test = train.copy(), test.copy()

    medians = train[num_cols].______()
    train[num_cols] = train[num_cols].fillna(medians)
    test[num_cols] = test[num_cols].fillna(______)

    mu = train[num_cols].mean()
    sigma = train[num_cols].std()
    train[num_cols] = (train[num_cols] - mu) / sigma
    test[num_cols] = (test[num_cols] - ______) / ______

    X_train = pd.get_dummies(train[num_cols + [cat_col]], columns=[cat_col], dtype=int)
    X_test = pd.get_dummies(test[num_cols + [cat_col]], columns=[cat_col], dtype=int)
    X_test = X_test.reindex(columns=______, fill_value=0)
    return X_train, X_test

tr = pd.DataFrame({"x": [1.0, 2.0, np.nan, 5.0], "c": ["a", "b", "a", "a"]})
te = pd.DataFrame({"x": [np.nan, 8.0], "c": ["b", "z"]})
X_train, X_test = prepare(tr, te, ["x"], "c")
print(X_train.round(2))
print(X_test.round(2))
```

<div dir="rtl">

الف) جاهای خالی را کامل کنید.

ب) خروجی را حساب کنید. (میانه‌ی <code dir="ltr">x</code> در آموزش را پیدا کنید، جای خالی را پر کنید، بعد میانگین و انحراف معیار را حساب کنید.)

ج) چرا پر کردن گمشده‌ها **قبل از** استانداردسازی انجام می‌شود؟ اگر ترتیب برعکس بود چه فرقی می‌کرد؟

<details>
<summary><b>پاسخ</b></summary>

الف)

<div dir="ltr">

```python
medians = train[num_cols].median()
test[num_cols] = test[num_cols].fillna(medians)
test[num_cols] = (test[num_cols] - mu) / sigma
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
```

</div>

ب) میانه‌ی <code dir="ltr">[1, 2, 5]</code> برابر <code dir="ltr">2</code> است؛ ستون آموزش <code dir="ltr">[1, 2, 2, 5]</code> می‌شود با میانگین <code dir="ltr">2.5</code> و انحراف معیار <code dir="ltr">1.73</code>. نتیجه:

<div dir="ltr">

```text
      x  c_a  c_b
0 -0.87    1    0
1 -0.29    0    1
2 -0.29    1    0
3  1.44    1    0
      x  c_a  c_b
0 -0.29    0    1
1  3.18    0    0
```

</div>

سطر اول آزمون با میانه‌ی **آموزش** (<code dir="ltr">2</code>) پر شده و همان <code dir="ltr">-0.29</code> را گرفته است. سطر دوم دسته‌ی ناشناخته‌ی <code dir="ltr">z</code> دارد و بردار دسته‌ای تمام‌صفر گرفته است؛ مقدار <code dir="ltr">8</code> با آماره‌های آموزش به <code dir="ltr">3.18</code> تبدیل شده (بزرگ، اما همان چیزی که باید باشد).

ج) استانداردسازی به میانگین و انحراف معیار نیاز دارد؛ اگر اول استاندارد کنید، <code dir="ltr">mean</code> و <code dir="ltr">std</code> در Pandas گمشده‌ها را نادیده می‌گیرند و بعد پر کردن با میانه‌ی **استانداردشده** هم کار می‌کند، اما ستون حاصل دیگر دقیقاً میانگین صفر و انحراف معیار یک ندارد (مقدارهای پرشده اضافه شده‌اند). ترتیب «پر کن، بعد مقیاس کن» تضمین می‌کند داده‌ای که مدل می‌بیند همان آماره‌هایی را دارد که مقیاس‌بندی وعده داده است. در عمل این مراحل را در یک **pipeline** با ترتیب ثابت قرار می‌دهند تا روی آموزش و آزمون دقیقاً یکسان اجرا شوند.

</details>

</div>

<div dir="rtl">

## خلاصه‌ی جلسه

- گمشده‌ها را اول **اندازه** بگیرید و بپرسید چرا گمشده‌اند؛ بعد بین حذف (<code dir="ltr">dropna</code> با <code dir="ltr">subset</code> و <code dir="ltr">thresh</code>)، پر کردن (ثابت، میانه، گروهی با <code dir="ltr">transform</code>، <code dir="ltr">ffill</code>) و ستون نشانگر انتخاب کنید.
- پرت‌ها را با قاعده‌ی IQR (مقاوم) پیدا کنید؛ z-score در حضور پرت‌ها دچار اثر پوشاندن می‌شود. پرت همیشه نسبت به یک گروه مرجع معنی دارد و همیشه خطا نیست.
- با <code dir="ltr">to&#95;numeric</code> و <code dir="ltr">to&#95;datetime</code> (با <code dir="ltr">errors="coerce"</code>) و متدهای <code dir="ltr">str</code>، قالب را درست کنید و با قاعده‌های دامنه، معنا را.
- دسته‌های اسمی را one-hot و دسته‌های ترتیبی را با <code dir="ltr">map</code> کدگذاری کنید؛ مراقب دام متغیر مجازی و دسته‌های ناشناخته باشید.
- مقیاس‌ها را با min-max یا z-score یکسان کنید و **همه‌ی** آماره‌های آماده‌سازی را فقط از داده‌ی آموزش یاد بگیرید.

**جلسه‌ی بعد:** مصورسازی با Matplotlib؛ رسم نمودارهای پایه، <code dir="ltr">subplot</code> و شخصی‌سازی و ذخیره‌ی نمودار.

</div>